# Statistical Process Control (SPC) for Manufacturing Quality

**Goal:** monitor a machined part's critical dimension and decide whether the production process is *in control* and *capable* of meeting specification — the core daily work of a manufacturing QA/QC department.

**Methods:** X-bar & R control charts, Western Electric run rules, and process-capability indices (Cp / Cpk).

> **Note on data:** the measurement data is *synthetically generated* to simulate a realistic machining process (target 10.00 mm, spec 9.95–10.05 mm) with a deliberate tool-wear shift partway through. This is a methods demonstration, not real company data.

## 1. Load the measurement data
Subgroups of 5 parts measured across 40 time periods.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/measurements.csv')
USL, LSL, TARGET = 10.05, 9.95, 10.00   # specification limits (mm)
print(df.head())
print(f"\n{len(df)} measurements, {df.subgroup.nunique()} subgroups of {df.groupby('subgroup').size().iloc[0]}")

## 2. Subgroup statistics
For each subgroup we compute the **mean (X-bar)** and **range (R)**. Control charts track these two things: the mean tells us if the process is centered; the range tells us if its variation is stable.

In [ ]:
g = df.groupby('subgroup')['measurement_mm']
stats = pd.DataFrame({'xbar': g.mean(), 'R': g.max()-g.min()}).reset_index()
stats.head()

## 3. Control limits
Using standard Shewhart constants for subgroup size n=5 (A2=0.577, D4=2.114, D3=0). Control limits come **from the process data itself** — they describe what the process naturally does, which is different from the specification limits (what we *want* it to do).

In [ ]:
A2, D3, D4, d2 = 0.577, 0.0, 2.114, 2.326
xbarbar = stats.xbar.mean()
Rbar = stats.R.mean()
xbar_UCL, xbar_LCL = xbarbar + A2*Rbar, xbarbar - A2*Rbar
R_UCL = D4*Rbar

print(f"X-bar center: {xbarbar:.4f}  UCL: {xbar_UCL:.4f}  LCL: {xbar_LCL:.4f}")
print(f"R center: {Rbar:.4f}  UCL: {R_UCL:.4f}")

## 4. X-bar control chart
Points outside the red limits, or long runs on one side, signal the process has changed.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(stats.subgroup, stats.xbar, '-o', color='#4A7BA7', markersize=4)
ax.axhline(xbarbar, color='green', label='Center')
ax.axhline(xbar_UCL, color='#C0392B', ls='--', label='Control limits')
ax.axhline(xbar_LCL, color='#C0392B', ls='--')
ooc = stats[(stats.xbar>xbar_UCL)|(stats.xbar<xbar_LCL)]
ax.scatter(ooc.subgroup, ooc.xbar, color='#C0392B', s=60, zorder=5, label='Out of control')
ax.set_xlabel('Subgroup'); ax.set_ylabel('Sample mean (mm)')
ax.set_title('X-bar Control Chart'); ax.legend()
plt.tight_layout(); plt.show()
print("Out-of-control subgroups:", list(ooc.subgroup))

## 5. Detecting a process shift with run rules
Even when individual points stay inside the limits, a long **run** of points on one side of the center line signals a shift. Here a run appears after the tool-wear point — the kind of early warning SPC is designed to catch.

In [ ]:
side = np.sign(stats.xbar.values - xbarbar)
start = 0
for i in range(1, len(side)+1):
    if i==len(side) or side[i]!=side[start]:
        if i-start >= 8:
            print(f"Run of {i-start} points: subgroups "
                  f"{stats.subgroup.iloc[start]}-{stats.subgroup.iloc[i-1]}")
        start = i

## 6. Process capability: Cp and Cpk
- **Cp** = tolerance width / process spread → is the process *tight enough*?
- **Cpk** also accounts for *centering* → is it tight enough **and** on target?

Industry commonly targets **Cpk ≥ 1.33**.

In [ ]:
sigma = Rbar / d2
Cp  = (USL - LSL) / (6*sigma)
Cpk = min((USL - xbarbar), (xbarbar - LSL)) / (3*sigma)
print(f"Process sigma: {sigma:.4f} mm")
print(f"Cp  = {Cp:.2f}")
print(f"Cpk = {Cpk:.2f}  -> {'PASS' if Cpk>=1.33 else 'below 1.33: process off-center, needs attention'}")

## 7. Capability histogram
The distribution against the spec limits — a visual read on how much margin the process has.

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(df.measurement_mm, bins=25, color='#4A7BA7', alpha=0.7, edgecolor='white')
for x,c,l in [(USL,'#C0392B','USL'),(LSL,'#C0392B','LSL'),(TARGET,'green','Target')]:
    ax.axvline(x, color=c, ls='--', label=l)
ax.set_xlabel('Measurement (mm)'); ax.set_ylabel('Count')
ax.set_title(f'Capability: Cp={Cp:.2f}, Cpk={Cpk:.2f}'); ax.legend()
plt.tight_layout(); plt.show()

## Summary

The control charts detect the injected tool-wear shift (out-of-control points and a sustained run after subgroup 26), and the capability analysis shows a process that is **precise but off-center**: Cp ≈ 1.43 (spread is fine) yet Cpk ≈ 1.25 (below the 1.33 target because the mean drifted). In a real QA setting this is the trigger to investigate and re-center the process before defects accumulate.

**What this demonstrates:** reading measurement data, applying SPC correctly, distinguishing control limits from specification limits, and interpreting capability indices — the everyday toolkit of manufacturing quality assurance.

**Honest scope:** synthetic data; single characteristic; standard Shewhart charts. Natural extensions: multiple characteristics, attribute charts (p/np) for pass-fail data, and automated out-of-control alerts.